# Pandas — Day 7 Practical

### Python Course · Colab Notebook

---

**How to use this notebook**

1. Run every cell in order, top to bottom. Click the cell and press **Shift + Enter**.
2. **Every single line of code has a comment** explaining what it does. Read the comment, then read the line.
3. Each section ends with a **YOUR TURN** cell. Those are the exercises — do them.
4. Nothing to upload: section 2 writes its own CSV file and reads it straight back.

**To save your own copy:** File → Save a copy in Drive.

---

### What we are covering

| Section | Topic |
|---|---|
| 1 | Series and DataFrame — the only two objects |
| 2 | Loading data, and the first five checks |
| 3 | Selecting columns |
| 4 | `.loc` and `.iloc` |
| 5 | Filtering rows |
| 6 | The copy trap |
| 7 | New columns and missing data |
| 8 | sort · value_counts · groupby |
| 9 | merge and saving to CSV |
| ★ | Mini project — the class report |

---

> **A note on versions.** Pandas changes over time, so your screen may differ from your neighbour's
> in four small, harmless ways:
>
> - text columns are labelled `object` on Pandas 2 and `str` on Pandas 3
> - `df.info()` prints a slightly different first line and memory figure
> - Section 6 prints a `SettingWithCopyWarning` on Pandas 2 and stays silent on Pandas 3
> - the version numbers themselves, obviously
>
> **Every number, every table and every lesson in this notebook is the same on both.**
> The first cell prints your version.

---
# Section 1 — The only two objects

Everything in Pandas is a **Series** (one column) or a **DataFrame** (a whole table).
Learn the pair and the rest is method names.

In [79]:
import pandas as pd            # load the Pandas library and call it "pd" from now on (everyone does this)
import numpy as np             # load NumPy too - Pandas is built on top of it and we will use np.where later

print("pandas version:", pd.__version__)   # print which version Colab gave you
print("numpy version :", np.__version__)   # and which NumPy - both are already installed, nothing to pip install

pandas version: 2.2.3
numpy version : 2.1.3


In [80]:
# ---- A SERIES is one column of data ----

s = pd.Series([78, 55, 91, 42])   # pd.Series( ) takes a list and turns it into a labelled column

print(s)                          # printing shows two columns: the INDEX on the left, the values on the right

0    78
1    55
2    91
3    42
dtype: int64


Look carefully at that output. The `0 1 2 3` down the left is **not** part of your data — it is the
**index**, a label for each row that Pandas added for you. Underneath, the values are a plain NumPy array.

> **A Series is an array of values plus an index of labels.** For number columns that array really is
> a NumPy `ndarray`, as the next cell shows. (For *text* columns, newer Pandas uses its own string
> array instead — same idea, slightly different container.)

In [81]:
print(s.values)                   # .values gives you the raw array underneath - just values, no labels
print(type(s.values).__name__)    # "ndarray" for a number column - it really IS a NumPy array
                                  # (a TEXT column may say StringArray on newer Pandas - same idea, different box)

print(list(s.index))              # .index gives the labels; list( ) makes them print tidily

[78 55 91 42]
ndarray
[0, 1, 2, 3]


In [82]:
# The index does not have to be 0,1,2. You can label rows with anything.

named = pd.Series([78, 55, 91],                      # the values
                  index=["Aisha", "Ben", "Carla"])   # index= gives each value a name instead of a number

print(named)                      # now the left-hand column shows names
print()                           # a blank line to separate the outputs

print(named["Ben"])               # look a value up BY ITS LABEL - this is what a plain array cannot do
print(named.mean())               # every NumPy aggregation still works: mean, sum, min, max, std

Aisha    78
Ben      55
Carla    91
dtype: int64

55
74.66666666666667


### ⭐ YOUR TURN 1

Make a Series called `temps` holding `18, 22, 31, 27`, with the index `"Mon", "Tue", "Wed", "Thu"`.
Then print Wednesday's temperature and the average of all four.

In [83]:
# YOUR TURN 1 - write your code here

temps = pd.Series([18, 22, 31, 27 ], index=["Mon", "Tue", "Wed", "Thu" ])   # TODO 1: fill in the values and the index
print(temps)                                # TODO 2: look at the whole thing
print(temps["Wed"])                         # TODO 3: one value, by its label
print(temps.mean())                         # TODO 4: the average

Mon    18
Tue    22
Wed    31
Thu    27
dtype: int64
31
24.5


---
# Section 2 — Loading data

Two ways in: build it from a dictionary, or read a file.

In [84]:
# ---- WAY 1: from a dictionary ----
# The KEYS become the column names. Each LIST becomes one column.

students = pd.DataFrame({                                                        # pd.DataFrame( ) takes a dictionary
    "name":    ["Aisha", "Ben", "Carla", "Dev", "Eli", "Farah", "Gita", "Hari"], # eight students
    "branch":  ["CSE", "ECE", "CSE", "MECH", "ECE", "CSE", "MECH", "ECE"],       # which branch each one is in
    "year":    [2, 2, 3, 2, 3, 3, 2, 3],                                         # which year of the course
    "maths":   [78, 55, 91, 42, 67, 83, 71, 60],                                 # marks out of 100
    "physics": [65, 71, 88, 38, 74, 97, 58, 82],                                 # more marks
    "english": [82, 60, 79, 55, 71, 68, 90, 64],                                 # more marks
})                                                                               # the } closes the dict, the ) the call

print(students)      # printing a DataFrame lays it out as a table, with the index down the left

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
1    Ben    ECE     2     55       71       60
2  Carla    CSE     3     91       88       79
3    Dev   MECH     2     42       38       55
4    Eli    ECE     3     67       74       71
5  Farah    CSE     3     83       97       68
6   Gita   MECH     2     71       58       90
7   Hari    ECE     3     60       82       64


> **Every list must be the same length.** Give one list seven items and the rest eight, and you get
> a `ValueError`. Pandas cannot build a table with a ragged edge.

In [85]:
# ---- WAY 2: from a CSV file ----
# First we WRITE one, so there is nothing for you to upload.

students.to_csv("students.csv", index=False)   # .to_csv( ) saves the table as a text file
                                               # index=False stops it writing the 0,1,2 column into the file

df = pd.read_csv("students.csv")               # read it straight back in - THIS is the line you will type most often

print(df.head())                               # .head( ) shows the first five rows so you can eyeball it

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
1    Ben    ECE     2     55       71       60
2  Carla    CSE     3     91       88       79
3    Dev   MECH     2     42       38       55
4    Eli    ECE     3     67       74       71


`read_csv` did three things for you without being asked:

1. **Read the header.** The first line of the file became your column names.
2. **Worked out the types.** Digits became whole numbers, text stayed text.
3. **Handled gaps.** An empty cell would have become `NaN` rather than crashing.

In Colab, to use your *own* file you would either run `files.upload()` from `google.colab`,
or pass a web address straight to `read_csv`.

### The five things you do to every new file

In this order. Every single time. Before you touch anything.

In [86]:
print(df.head(3))     # 1. THE FIRST ROWS - does this look like what you expected?
                      #    head( ) gives five by default; head(3) gives three

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
1    Ben    ECE     2     55       71       60
2  Carla    CSE     3     91       88       79


In [87]:
print(df.shape)       # 2. THE SIZE - a tuple of (rows, columns). Is it as big as you thought?
                      #    Same .shape attribute you used in NumPy yesterday

(8, 6)


In [88]:
print(df.columns)     # 3. THE COLUMN NAMES - exactly how are they spelled?
                      #    Nearly every Pandas error is a name typed slightly wrong

print(list(df.columns))   # list( ) prints them more readably than the raw Index object

Index(['name', 'branch', 'year', 'maths', 'physics', 'english'], dtype='object')
['name', 'branch', 'year', 'maths', 'physics', 'english']


In [89]:
df.info()             # 4. TYPES AND GAPS - note there is no print( ), info( ) prints by itself
                      #    Read the "Non-Null Count" column: 8 non-null out of 8 rows means nothing is missing
                      #    For TEXT columns the Dtype says "object" on Pandas 2 and "str" on Pandas 3.
                      #    They are not identical under the hood, but for today they mean the same: "this is text"

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   name     8 non-null      object
 1   branch   8 non-null      object
 2   year     8 non-null      int64 
 3   maths    8 non-null      int64 
 4   physics  8 non-null      int64 
 5   english  8 non-null      int64 
dtypes: int64(4), object(2)
memory usage: 516.0+ bytes


In [90]:
print(df.describe())  # 5. THE NUMBERS - count, mean, std, min, quartiles and max
                      #    It quietly SKIPS the text columns, because you cannot average a name

           year      maths    physics    english
count  8.000000   8.000000   8.000000   8.000000
mean   2.500000  68.375000  71.625000  71.125000
std    0.534522  15.909903  18.461833  11.837441
min    2.000000  42.000000  38.000000  55.000000
25%    2.000000  58.750000  63.250000  63.000000
50%    2.500000  69.000000  72.500000  69.500000
75%    3.000000  79.250000  83.500000  79.750000
max    3.000000  91.000000  97.000000  90.000000


> **`df.info()` is the important one.** It is the only one of the five that tells you how many
> values are **missing**, and missing values are what break everything later on.

> **`df.tail()`** shows the last five rows. Worth a look too — bad rows often hide at the bottom of a file.

---
# Section 3 — Selecting columns

One bracket or two. This catches everybody once.

In [91]:
maths_col = df["maths"]              # ONE bracket, one column name -> you get a SERIES back

print(maths_col)                     # note the "Name: maths" line at the bottom - that only appears on a Series
print(type(maths_col).__name__)      # "Series" - .__name__ just prints the type's name tidily

0    78
1    55
2    91
3    42
4    67
5    83
6    71
7    60
Name: maths, dtype: int64
Series


In [92]:
two_cols = df[["name", "maths"]]     # TWO brackets. The inner [ ] is a LIST of names; the outer [ ] is the lookup.

print(two_cols)                      # you get a DATAFRAME back - it prints as a table
print(type(two_cols).__name__)       # "DataFrame"

    name  maths
0  Aisha     78
1    Ben     55
2  Carla     91
3    Dev     42
4    Eli     67
5  Farah     83
6   Gita     71
7   Hari     60
DataFrame


In [93]:
one_col_frame = df[["maths"]]        # TWO brackets with only ONE name inside

print(type(one_col_frame).__name__)  # still "DataFrame" - the brackets decide the type, not how many names

DataFrame


### Why the difference matters

- A **Series** has `.mean()`, `.value_counts()`, `.unique()` — the one-column tools.
- A **DataFrame** prints as a table, can hold several columns, and can be saved to CSV.

Ask for the wrong one and the method you want will not exist. That is the single most common
Pandas confusion, and now you will not have it.

In [94]:
print(df["maths"].mean())            # works - a Series has .mean( )
print(df["maths"].max())             # so does .max( )
print(df["branch"].unique().tolist())  # .unique( ) lists the distinct values; .tolist( ) makes it a plain Python list
print(df["branch"].nunique())        # .nunique( ) just counts how many distinct values there are

68.375
91
['CSE', 'ECE', 'MECH']
3


### ⭐ YOUR TURN 2

1. Print the `physics` column on its own.
2. Print a two-column table of `name` and `physics`.
3. Print the average physics mark.

In [95]:
# YOUR TURN 2 - write your code here

print(df["physics" ])            # TODO 1: one bracket, one name
print(df[[ "name", "physics" ]])        # TODO 2: two brackets, two names in a list
print(df["physics"].mean())     # TODO 3: get the column first, then ask it for its mean

0    65
1    71
2    88
3    38
4    74
5    97
6    58
7    82
Name: physics, dtype: int64
    name  physics
0  Aisha       65
1    Ben       71
2  Carla       88
3    Dev       38
4    Eli       74
5  Farah       97
6   Gita       58
7   Hari       82
71.625


---
# Section 4 — `.loc` and `.iloc`

Two ways to pick rows. This is the part people get wrong, so we will be slow about it.

- **`.iloc`** — by **position**. The `i` is for integer. Exactly like NumPy.
- **`.loc`** — by **label**. The row's index label, and the column's name.

In [96]:
# ---- .iloc : by POSITION ----

print(df.iloc[0])         # row at position 0 - the FIRST row, returned as a Series
                          # the column names run down the left, because a single row turned sideways is a Series

name       Aisha
branch       CSE
year           2
maths         78
physics       65
english       82
Name: 0, dtype: object


In [97]:
print(df.iloc[0:3])       # rows at positions 0, 1, 2 - the stop is EXCLUDED, exactly like list slicing

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
1    Ben    ECE     2     55       71       60
2  Carla    CSE     3     91       88       79


In [98]:
print(df.iloc[0, 3])      # [row position, column position] -> row 0, column 3
                          # counting the columns: 0=name, 1=branch, 2=year, 3=maths -> so this is 78

print(df.iloc[-1])        # -1 counts from the end, so this is the LAST row

78
name       Hari
branch      ECE
year          3
maths        60
physics      82
english      64
Name: 7, dtype: object


In [99]:
print(df.iloc[0:3, 0:2])  # a rectangle: rows 0-2 and columns 0-1
                          # both stops are excluded, so you get 3 rows and 2 columns

    name branch
0  Aisha    CSE
1    Ben    ECE
2  Carla    CSE


### `.loc` — by label

In [100]:
print(df.loc[0, "maths"])          # [row LABEL, column NAME] - much easier to read than iloc[0, 3]
                                   # you do not have to count columns, you just say what you want

78


In [101]:
print(df.loc[0:2, ["name", "maths"]])   # rows labelled 0 to 2, and a LIST of column names

    name  maths
0  Aisha     78
1    Ben     55
2  Carla     91


### ⚠️ The one surprise: `.loc` **includes** the stop

`df.iloc[0:3]` gives you three rows — positions 0, 1, 2. Normal Python slicing.

`df.loc[0:2]` also gives you three rows — labels 0, **1 and 2**. The stop label is **included**.

This is the only slice in Python that behaves this way. The reason: a label is a *name*, not a count.
"Everything from row 0 to row 2" naturally means all three.

In [102]:
print("iloc[0:3] gives", len(df.iloc[0:3]), "rows")   # len( ) counts the rows - 3, stop excluded
print("loc[0:2]  gives", len(df.loc[0:2]), "rows")    # also 3 - but note we asked for 0 to 2, not 0 to 3

iloc[0:3] gives 3 rows
loc[0:2]  gives 3 rows


### Why they come apart

Right now the labels *are* `0, 1, 2, 3...`, so `.loc` and `.iloc` look almost identical.
Sort the table and they separate immediately — because **the labels stay glued to their rows**
while the positions get shuffled.

In [103]:
sorted_df = df.sort_values("maths", ascending=False)   # sort by maths, biggest first
                                                       # ascending=False means high-to-low
                                                       # (sorting gets its own section later - we only
                                                       #  need it here to shuffle the index)

print(sorted_df[["name", "maths"]])   # look at the INDEX on the left - it is jumbled, not 0,1,2,3
                                      # Carla was row 2 in the original and she still carries the label 2

    name  maths
2  Carla     91
5  Farah     83
0  Aisha     78
6   Gita     71
4    Eli     67
7   Hari     60
1    Ben     55
3    Dev     42


In [104]:
print("iloc[0] - the TOP row:")           # a label so the output is readable
print(sorted_df.iloc[0]["name"])          # position 0 = whoever is now at the top = Carla

print("loc[0] - the row LABELLED 0:")     # another label
print(sorted_df.loc[0]["name"])           # label 0 = Aisha, who is now sitting in the middle somewhere

iloc[0] - the TOP row:
Carla
loc[0] - the row LABELLED 0:
Aisha


> **Rule of thumb: use `.loc` almost always.** Reach for `.iloc` only when you genuinely mean
> "the third row from the top", which is rarer than you would think.

---
# Section 5 — Filtering rows

Good news: this is **exactly** yesterday's boolean masking. Same symbols, same brackets, same rules.

In [105]:
mask = df["maths"] > 70      # a comparison on a column gives one True/False PER ROW, not a single answer

print(mask)                  # a Series of True and False, the same length as the table
print(mask.dtype)            # bool - it is a real Series, just holding True/False instead of numbers

0     True
1    False
2     True
3    False
4    False
5     True
6     True
7    False
Name: maths, dtype: bool
bool


In [106]:
print(df[mask])              # put the True/False Series inside the brackets to keep only the True rows

print()                      # blank line
print(df[df["maths"] > 70])  # the same thing written in one line - this is the version you will use

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
2  Carla    CSE     3     91       88       79
5  Farah    CSE     3     83       97       68
6   Gita   MECH     2     71       58       90

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
2  Carla    CSE     3     91       88       79
5  Farah    CSE     3     83       97       68
6   Gita   MECH     2     71       58       90


Look at the **index** on the left of that result: `0, 2, 5, 6`. Rows 1, 3, 4 and 7 are gone, and the
labels were **not renumbered**. That is how you can still tell which original rows survived.

In [107]:
# ---- two conditions ----

print(df[(df["maths"] > 70) & (df["branch"] == "CSE")])   # & means BOTH must be true
                                                          # note the brackets around EACH condition - compulsory

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
2  Carla    CSE     3     91       88       79
5  Farah    CSE     3     83       97       68


In [108]:
print(df[(df["maths"] > 85) | (df["physics"] > 85)])      # | means EITHER one is enough

    name branch  year  maths  physics  english
2  Carla    CSE     3     91       88       79
5  Farah    CSE     3     83       97       68


In [109]:
print(df[~(df["branch"] == "CSE")])   # ~ flips every True to False, so this is "everyone NOT in CSE"

   name branch  year  maths  physics  english
1   Ben    ECE     2     55       71       60
3   Dev   MECH     2     42       38       55
4   Eli    ECE     3     67       74       71
6  Gita   MECH     2     71       58       90
7  Hari    ECE     3     60       82       64


In [110]:
print(df[df["branch"].isin(["CSE", "MECH"])])   # .isin( ) checks against a LIST of allowed values
                                                # this saves writing  (b == "CSE") | (b == "MECH")

    name branch  year  maths  physics  english
0  Aisha    CSE     2     78       65       82
2  Carla    CSE     3     91       88       79
3    Dev   MECH     2     42       38       55
5  Farah    CSE     3     83       97       68
6   Gita   MECH     2     71       58       90


In [111]:
print("how many scored over 70:", (df["maths"] > 70).sum())   # True counts as 1, False as 0, so .sum( ) COUNTS
print("did anyone score over 90:", (df["maths"] > 90).any())  # .any( ) is True if at least one is True
print("did everyone pass 40:    ", (df["maths"] > 40).all())  # .all( ) is True only if every single one is True

how many scored over 70: 4
did anyone score over 90: True
did everyone pass 40:     True


> **`&` `|` `~` — never `and` `or` `not`.** The words want one True/False; here you have eight.
> If you see the word **"ambiguous"** in the error, that is what happened — or you forgot a bracket.

### ⭐ YOUR TURN 3

1. Everyone in year 3.
2. Everyone in ECE who scored above 60 in physics.
3. How many students are in CSE?

In [112]:
print(df[df["year" ] == 3])                          # TODO 1: one condition
print(df[(df["branch"] == "ECE") & (df["physics"] > 60)])     # TODO 2: two conditions, brackets round each
print((df["branch"] == "CSE").sum())                  # TODO 3: build the mask, then .sum( ) it

    name branch  year  maths  physics  english
2  Carla    CSE     3     91       88       79
4    Eli    ECE     3     67       74       71
5  Farah    CSE     3     83       97       68
7   Hari    ECE     3     60       82       64
   name branch  year  maths  physics  english
1   Ben    ECE     2     55       71       60
4   Eli    ECE     3     67       74       71
7  Hari    ECE     3     60       82       64
3


---
# Section 6 — The copy trap ⚠️

Yesterday you learned that a NumPy **slice** (`arr[1:4]`) is a *view* — edit it and the original changes.
You also learned that a NumPy **mask** (`arr[arr > 60]`) is a *copy*.

Pandas filtering is a mask, so it gives you a **copy** — consistent with yesterday. The trap is that
the copy looks and behaves exactly like the original right up until you try to change it, and then
your edit quietly goes nowhere.

In [113]:
cse = df[df["branch"] == "CSE"]    # filter down to the CSE students - this builds a NEW little table

cse["maths"] = 100                 # set every maths mark in THAT table to 100
                                   # (older Pandas prints a SettingWithCopyWarning here; newer versions say nothing)

print("the filtered table:")       # a label
print(cse[["name", "maths"]])      # inside cse, the marks really are 100

the filtered table:
    name  maths
0  Aisha    100
2  Carla    100
5  Farah    100


/tmp/ipykernel_971/1835814510.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cse["maths"] = 100                 # set every maths mark in THAT table to 100


In [114]:
print("the original table:")            # a label
print(df[["name", "maths"]].head(3))    # ...but the original is completely untouched

print("Aisha's maths in df is still:", df.loc[0, "maths"])   # 78, not 100

the original table:
    name  maths
0  Aisha     78
1    Ben     55
2  Carla     91
Aisha's maths in df is still: 78


### What happened

The filter did not hand you a window onto `df`. It built a **brand new little table** and gave you that.
You edited the new table. The original never heard about it.

There are two fixes, and which one you want depends on what you were trying to do.

In [115]:
# ---- FIX 1: you genuinely want a separate table - say so with .copy( ) ----

cse_copy = df[df["branch"] == "CSE"].copy()   # .copy( ) makes it explicit: this is mine, it is separate

cse_copy["maths"] = 100                       # now nobody is confused about what you meant

print(cse_copy[["name", "maths"]])            # the copy has 100s
print("original untouched:", df.loc[0, "maths"])   # and df is still 78, which is what we wanted

    name  maths
0  Aisha    100
2  Carla    100
5  Farah    100
original untouched: 78


In [116]:
# ---- FIX 2: you want to change the ORIGINAL - do it in one .loc ----

df2 = df.copy()                                          # work on a copy of df so the rest of the notebook is unaffected

df2.loc[df2["branch"] == "CSE", "maths"] = 100           # .loc[which rows, which column] = new value
                                                         # one step: pick the rows AND the column AND assign, together

print(df2[["name", "branch", "maths"]])                  # the CSE students really did change this time

    name branch  maths
0  Aisha    CSE    100
1    Ben    ECE     55
2  Carla    CSE    100
3    Dev   MECH     42
4    Eli    ECE     67
5  Farah    CSE    100
6   Gita   MECH     71
7   Hari    ECE     60


> **The rule**
>
> - Filtering to **look** at something → just filter.
> - Filtering to **change** something → either `.copy()` it first, or do it in one `df.loc[mask, "col"] = value`.
>
> Fix 2 is what you will see in professional code. One `.loc`, one step, no ambiguity.

---
# Section 7 — New columns and missing data

In [117]:
df["total"] = df["maths"] + df["physics"] + df["english"]   # assign to a name that does not exist yet and it APPEARS
                                                            # the addition is element-wise: row 0 + row 0, row 1 + row 1

df["average"] = (df["total"] / 3).round(1)                  # divide the whole column by 3, then .round(1) to 1 decimal

df["result"] = np.where(df["average"] >= 60, "pass", "fail")   # yesterday's np.where: condition, if-true, if-false
                                                               # it takes a Pandas Series just as happily as an array

print(df[["name", "total", "average", "result"]])           # show just the four columns we care about

    name  total  average result
0  Aisha    225     75.0   pass
1    Ben    186     62.0   pass
2  Carla    258     86.0   pass
3    Dev    135     45.0   fail
4    Eli    212     70.7   pass
5  Farah    248     82.7   pass
6   Gita    219     73.0   pass
7   Hari    206     68.7   pass


In [118]:
df_dropped = df.drop(columns=["english"])   # .drop( ) removes columns; it returns a NEW table

print(list(df_dropped.columns))             # english is gone from the new one
print(list(df.columns))                     # ...but still present in the original, because drop did not change it

['name', 'branch', 'year', 'maths', 'physics', 'total', 'average', 'result']
['name', 'branch', 'year', 'maths', 'physics', 'english', 'total', 'average', 'result']


In [119]:
df_renamed = df.rename(columns={"maths": "Maths"})   # rename takes a dictionary: {old name: new name}

print(list(df_renamed.columns))                      # "maths" is now "Maths" - capital M

['name', 'branch', 'year', 'Maths', 'physics', 'english', 'total', 'average', 'result']


### Missing data — `NaN`

Every real dataset has holes. Pandas marks them **NaN** ("not a number") and you have to decide
what to do about each one.

In [120]:
messy = pd.DataFrame({                              # a small table with deliberate gaps
    "name": ["Aisha", "Ben", "Carla", "Dev"],       # four names, none missing
    "maths": [78, np.nan, 91, 42],                  # np.nan is the missing-value marker - Ben's mark is absent
    "physics": [65, 71, np.nan, 38],                # Carla's physics mark is absent
})                                                  # the } closes the dictionary, the ) closes DataFrame( )

print(messy)                                        # the gaps print as NaN

    name  maths  physics
0  Aisha   78.0     65.0
1    Ben    NaN     71.0
2  Carla   91.0      NaN
3    Dev   42.0     38.0


Notice the marks now print as `78.0` and `65.0` rather than `78` and `65`.

`NaN` is a **float**, so one gap drags the whole column to `float64` — the same type-dragging you saw
in NumPy yesterday. It is normal, not a mistake.

In [121]:
print(messy.isna())          # .isna( ) gives a True/False table - True wherever a value is missing

    name  maths  physics
0  False  False    False
1  False   True    False
2  False  False     True
3  False  False    False


In [122]:
print(messy.isna().sum())    # .sum( ) on that True/False table COUNTS the Trues, one count per column
                             # this two-word line is how you check any new file for gaps

name       0
maths      1
physics    1
dtype: int64


In [123]:
print(messy["maths"].mean())   # mean( ) SKIPS the NaN automatically - it does not crash, and it does
                               # not treat the gap as a zero: (78 + 91 + 42) / 3 = 70.33, NOT divided by 4
                               # (careful: sum( ) also skips NaN, but for a SUM that is the same answer
                               #  you would get by treating it as zero - the difference only shows in mean)

70.33333333333333


In [124]:
print(messy.dropna())        # .dropna( ) deletes every ROW that has any gap at all
                             # safe, but here it throws away half your data to fix two cells

    name  maths  physics
0  Aisha   78.0     65.0
3    Dev   42.0     38.0


In [125]:
print(messy.fillna(0))       # .fillna(0) puts 0 in every gap
                             # fast - but a 0 mark is a LIE. It says Ben scored nothing.

    name  maths  physics
0  Aisha   78.0     65.0
1    Ben    0.0     71.0
2  Carla   91.0      0.0
3    Dev   42.0     38.0


In [126]:
filled = messy.copy()                                            # work on a copy so `messy` stays messy for later

filled["maths"] = filled["maths"].fillna(filled["maths"].mean()) # fill the gaps in maths with the average OF maths
                                                                 # read it inside-out: work out the mean, then fill with it

print(filled)                                                    # Ben now has 70.333333, the average of the other three

    name      maths  physics
0  Aisha  78.000000     65.0
1    Ben  70.333333     71.0
2  Carla  91.000000      NaN
3    Dev  42.000000     38.0


> **Which one should you use?** There is no universal answer — it depends what the gap *means*.
>
> - `dropna()` — when a row with a gap is genuinely useless to you.
> - `fillna(0)` — only when zero is the honest answer (e.g. "number of purchases").
> - `fillna(mean)` — the usual sensible choice for a measurement that just was not recorded.
>
> What you must never do is ignore it and wonder later why your average looks wrong.

---
# Section 8 — Sorting, counting, grouping

In [127]:
print(df.sort_values("total", ascending=False)[["name", "total"]])   # sort by total, biggest first
                                                                     # then [[ ]] picks the two columns to display

    name  total
2  Carla    258
5  Farah    248
0  Aisha    225
6   Gita    219
4    Eli    212
7   Hari    206
1    Ben    186
3    Dev    135


In [128]:
print(df.sort_values(["branch", "total"],          # sort by TWO columns: branch first, then total within each branch
                     ascending=[True, False]       # branch A-Z (True), but total high-to-low (False)
                     )[["name", "branch", "total"]])   # show three columns

    name branch  total
2  Carla    CSE    258
5  Farah    CSE    248
0  Aisha    CSE    225
4    Eli    ECE    212
7   Hari    ECE    206
1    Ben    ECE    186
6   Gita   MECH    219
3    Dev   MECH    135


> `sort_values` returns a **new** table. It does not change `df`. If you want to keep the sorted
> version, save it: `df = df.sort_values(...)`.

In [129]:
print(df["branch"].value_counts())   # how many students in each branch - one word, and it sorts biggest first

branch
CSE     3
ECE     3
MECH    2
Name: count, dtype: int64


### `groupby` — the method that makes Pandas worth learning

*"For each branch, what is the average maths mark?"* — one line.

It works in three steps, always: **split** the table into piles, **apply** a calculation to each pile,
**combine** the answers back into one result.

In [130]:
print(df.groupby("branch")["maths"].mean().round(2))   # SPLIT by branch, APPLY mean to the maths column, COMBINE
                                                       # .round(2) just trims the decimals for readability

branch
CSE     84.00
ECE     60.67
MECH    56.50
Name: maths, dtype: float64


Read that result carefully. The word `branch` sits on its own line above the numbers because
**the grouping column has become the index** of the result.

In [131]:
# .agg( ) takes a LIST of what you want, and gives you one column for each:
# count = how many are in the group, mean = the average, max = the biggest
print(df.groupby("branch")["total"].agg(["count", "mean", "max"]).round(2))   # three answers per branch

        count    mean  max
branch                    
CSE         3  243.67  258
ECE         3  201.33  212
MECH        2  177.00  219


In [132]:
print(df.groupby(["branch", "year"])["total"].mean().round(1))   # group by TWO columns
                                                                 # you get one row per combination that exists

branch  year
CSE     2       225.0
        3       253.0
ECE     2       186.0
        3       209.0
MECH    2       177.0
Name: total, dtype: float64


### ⭐ YOUR TURN 4

1. The average `physics` mark per branch.
2. The number of students in each `year`.
3. The highest `total` in each branch.

In [133]:
# YOUR TURN 4 - write your code here

print(df.groupby('branch')['physics'].mean().round(2))   # TODO 1: group by branch, take physics, take the mean
print(df['year'].value_counts())               # TODO 2: value_counts on the year column
print(df.groupby('branch')['total'].max())             # TODO 3: same shape as TODO 1, but max instead of mean

branch
CSE     83.33
ECE     75.67
MECH    48.00
Name: physics, dtype: float64
year
2    4
3    4
Name: count, dtype: int64
branch
CSE     258
ECE     212
MECH    219
Name: total, dtype: int64


---
# Section 9 — Joining tables, and saving your work

In [134]:
branch_info = pd.DataFrame({                          # a second, small table
    "branch": ["CSE", "ECE", "MECH"],                 # the SHARED column - it must be spelled the same in both
    "hod": ["Dr Rao", "Dr Iyer", "Dr Shah"],          # the extra information we want to bring across
})                                                    # } closes the dictionary, ) closes the DataFrame( ) call

print(branch_info)                                    # three rows, two columns

  branch      hod
0    CSE   Dr Rao
1    ECE  Dr Iyer
2   MECH  Dr Shah


In [135]:
merged = df.merge(branch_info, on="branch")   # .merge( ) glues the two tables together on the shared column
                                              # on="branch" says which column to match rows up by

print(merged[["name", "branch", "hod"]])      # every student now carries their head of department

    name branch      hod
0  Aisha    CSE   Dr Rao
1    Ben    ECE  Dr Iyer
2  Carla    CSE   Dr Rao
3    Dev   MECH  Dr Shah
4    Eli    ECE  Dr Iyer
5  Farah    CSE   Dr Rao
6   Gita   MECH  Dr Shah
7   Hari    ECE  Dr Iyer


`Dr Rao` appears three times, because three students are in CSE. That is correct — the CSE row from
the small table was copied onto every matching student.

In [136]:
print("df shape    :", df.shape)          # (8, 9) - the original 6 columns plus total, average and result
print("merged shape:", merged.shape)      # (8, 10) - one extra column (hod), and the SAME number of rows

df shape    : (8, 9)
merged shape: (8, 10)


In [137]:
merged.to_csv("class_report.csv", index=False)   # save the finished table as a file
                                                 # index=False stops the 0,1,2 labels being written as a column

check = pd.read_csv("class_report.csv")          # read it straight back to prove it worked
print(check.head(3))                             # the first three rows of the saved file

    name branch  year  maths  physics  english  total  average result      hod
0  Aisha    CSE     2     78       65       82    225     75.0   pass   Dr Rao
1    Ben    ECE     2     55       71       60    186     62.0   pass  Dr Iyer
2  Carla    CSE     3     91       88       79    258     86.0   pass   Dr Rao


> **In Colab, where did that file go?** Into the temporary session storage. Click the folder icon in
> the left sidebar to see it, and the download arrow next to it to save it to your computer.
> It disappears when the session ends.

---
# ★ Mini project — the class report

Everything from today, on one table. Work through it cell by cell.

In [138]:
# ---- Start fresh from the file, the way a real analysis begins ----

report = pd.read_csv("students.csv")   # read the CSV we wrote back in Section 2

print(report.shape)                    # (8, 6) - eight students, six columns
print(list(report.columns))            # check the names before doing anything else

(8, 6)
['name', 'branch', 'year', 'maths', 'physics', 'english']


In [139]:
# ---- 1. Build the derived columns ----

report["total"] = report["maths"] + report["physics"] + report["english"]   # add the three subject columns together
report["average"] = (report["total"] / 3).round(1)                          # divide by 3 and round to 1 decimal

report["grade"] = np.where(report["average"] >= 80, "A",                    # 80 and above -> "A"
                  np.where(report["average"] >= 65, "B", "C"))              # otherwise 65+ -> "B", else "C"
                                                                            # this is an np.where INSIDE another one

print(report[["name", "total", "average", "grade"]])                        # show the result

    name  total  average grade
0  Aisha    225     75.0     B
1    Ben    186     62.0     C
2  Carla    258     86.0     A
3    Dev    135     45.0     C
4    Eli    212     70.7     B
5  Farah    248     82.7     A
6   Gita    219     73.0     B
7   Hari    206     68.7     B


In [140]:
# ---- 2. Who is top of the class? ----

ranked = report.sort_values("total", ascending=False)   # sort high to low

print(ranked[["name", "branch", "total"]].head(3))      # .head(3) gives the top three

print()                                                 # blank line
print("Top student:", ranked.iloc[0]["name"])           # iloc[0] is the top ROW after sorting; ["name"] pulls one value

    name branch  total
2  Carla    CSE    258
5  Farah    CSE    248
0  Aisha    CSE    225

Top student: Carla


In [141]:
# ---- 3. Per-branch summary ----

by_branch = report.groupby("branch")["total"].agg(["count", "mean", "max"]).round(1)   # three numbers per branch

print(by_branch)                                        # count, mean and max for CSE, ECE and MECH

        count   mean  max
branch                   
CSE         3  243.7  258
ECE         3  201.3  212
MECH        2  177.0  219


In [142]:
# ---- 4. Per-subject averages ----

subjects = ["maths", "physics", "english"]              # a list of the three subject column names

print(report[subjects].mean().round(2))                 # pick those three columns, then .mean( ) averages EACH column
                                                        # you get one number per subject

maths      68.38
physics    71.62
english    71.12
dtype: float64


In [143]:
# ---- 5. Who is at risk? (any single subject below 50) ----

at_risk = report[(report["maths"] < 50) |               # low in maths...
                 (report["physics"] < 50) |             # ...OR low in physics...
                 (report["english"] < 50)]              # ...OR low in english

print(at_risk[["name", "maths", "physics", "english"]]) # show their three marks so you can see why

  name  maths  physics  english
3  Dev     42       38       55


In [144]:
# ---- 6. Grade distribution ----

print(report["grade"].value_counts())                   # how many A, B and C grades were awarded

grade
B    4
C    2
A    2
Name: count, dtype: int64


In [145]:
# ---- 7. Bring in the head of department ----

final = report.merge(branch_info, on="branch")          # glue on the hod column from the small table

print(final[["name", "branch", "hod", "grade"]])        # a tidy four-column view

    name branch      hod grade
0  Aisha    CSE   Dr Rao     B
1    Ben    ECE  Dr Iyer     C
2  Carla    CSE   Dr Rao     A
3    Dev   MECH  Dr Shah     C
4    Eli    ECE  Dr Iyer     B
5  Farah    CSE   Dr Rao     A
6   Gita   MECH  Dr Shah     B
7   Hari    ECE  Dr Iyer     B


In [146]:
# ---- 8. The printed report ----

print("=" * 48)                                                   # "=" * 48 repeats the = sign to make a line
print("CLASS REPORT")                                             # a heading
print("=" * 48)                                                   # another line

ranked_final = final.sort_values("total", ascending=False)        # highest total first

for place in range(len(ranked_final)):                            # one turn per student, in ranking order
    row = ranked_final.iloc[place]                                # iloc[place] gives that whole row as a Series
    name = str(row["name"]).ljust(8)                              # pull the name out; .ljust(8) pads it so columns line up
    branch = str(row["branch"]).ljust(6)                          # same for the branch
    print(place + 1, name, branch, "total", row["total"], " grade", row["grade"])   # place + 1 because humans count from 1

print("-" * 48)                                                   # a divider
print("Class average :", round(float(report["average"].mean()), 2))   # the average of the averages
print("Highest total :", report["total"].max())                       # the single best total
print("Lowest total  :", report["total"].min())                       # and the worst
print("Grade A count :", (report["grade"] == "A").sum())              # count the Trues
print("=" * 48)                                                       # closing line

CLASS REPORT
1 Carla    CSE    total 258  grade A
2 Farah    CSE    total 248  grade A
3 Aisha    CSE    total 225  grade B
4 Gita     MECH   total 219  grade B
5 Eli      ECE    total 212  grade B
6 Hari     ECE    total 206  grade B
7 Ben      ECE    total 186  grade C
8 Dev      MECH   total 135  grade C
------------------------------------------------
Class average : 70.39
Highest total : 258
Lowest total  : 135
Grade A count : 2


In [147]:
# ---- 9. Save it ----

final.to_csv("final_report.csv", index=False)      # write the finished table to a file

print("saved. shape:", final.shape)                # confirm what we saved
print(pd.read_csv("final_report.csv").head(2))     # read the first two rows back as proof

saved. shape: (8, 10)
    name branch  year  maths  physics  english  total  average grade      hod
0  Aisha    CSE     2     78       65       82    225     75.0     B   Dr Rao
1    Ben    ECE     2     55       71       60    186     62.0     C  Dr Iyer


---
### ⭐ FINAL CHALLENGE

Add cells below and answer these, using `final`:

1. Which **branch** has the highest average total?
   *(Hint: `groupby(...)["total"].mean()` gives you a Series whose INDEX is the branch names. A Series
   has a method `.idxmax()` — "index of the max" — which hands back the label of the biggest value
   rather than the value itself. So the whole answer is one chained line.)*
2. Print every student whose `average` is above the class average.
   *(Hint: work out `report["average"].mean()` first, then use it in a filter.)*
3. Add a column `rank` numbering the students 1 to 8 by total, best first.
   *(Hint: sort `final` first and save the result, then assign `range(1, len(final) + 1)` to a new
   column on the sorted table.)*
4. Save a CSV containing only the year-3 students, sorted by total.

In [148]:
# FINAL CHALLENGE - your code here

# 1. Which branch has the highest average total?
print("1. Branch with the highest average total:")
highest_avg_total_branch = final.groupby('branch')['total'].mean().idxmax()
print(highest_avg_total_branch)
print()

# 2. Print every student whose average is above the class average.
print("2. Students with average above class average:")
class_average = report['average'].mean()
students_above_avg = final[final['average'] > class_average]
print(students_above_avg[['name', 'branch', 'average']])
print()

# 3. Add a column rank numbering the students 1 to 8 by total, best first.
print("3. Students with rank column:")
final_ranked = final.sort_values('total', ascending=False).copy()
final_ranked['rank'] = range(1, len(final_ranked) + 1)
print(final_ranked[['name', 'total', 'rank']])
print()

# 4. Save a CSV containing only the year-3 students, sorted by total.
print("4. Saving year-3 students to CSV:")
year_3_students = final[final['year'] == 3].sort_values('total', ascending=False)
year_3_students.to_csv('year_3_students_report.csv', index=False)
print("  'year_3_students_report.csv' created with the following data (first 5 rows):")
print(year_3_students.head())


1. Branch with the highest average total:
CSE

2. Students with average above class average:
    name branch  average
0  Aisha    CSE     75.0
2  Carla    CSE     86.0
4    Eli    ECE     70.7
5  Farah    CSE     82.7
6   Gita   MECH     73.0

3. Students with rank column:
    name  total  rank
2  Carla    258     1
5  Farah    248     2
0  Aisha    225     3
6   Gita    219     4
4    Eli    212     5
7   Hari    206     6
1    Ben    186     7
3    Dev    135     8

4. Saving year-3 students to CSV:
  'year_3_students_report.csv' created with the following data (first 5 rows):
    name branch  year  maths  physics  english  total  average grade      hod
2  Carla    CSE     3     91       88       79    258     86.0     A   Dr Rao
5  Farah    CSE     3     83       97       68    248     82.7     A   Dr Rao
4    Eli    ECE     3     67       74       71    212     70.7     B  Dr Iyer
7   Hari    ECE     3     60       82       64    206     68.7     B  Dr Iyer


---

## Ten lines to take home

1. A **DataFrame** is a spreadsheet you can program. A **Series** is one column of it.
2. A Series is a NumPy array **with labels** — everything from yesterday still applies underneath.
3. **One bracket** gives a Series, **two** give a DataFrame: `df["a"]` vs `df[["a", "b"]]`.
4. `.loc` is by **label**, `.iloc` is by **position**. Use `.loc` almost always.
5. `.loc` **includes** the stop label. It is the only slice in Python that does.
6. Filtering is yesterday's boolean masking — `&`, `|`, `~`, and brackets round every condition.
7. A filtered table is a **copy**. To change the original, use `df.loc[mask, "col"] = value`.
8. `df.info()` tells you what is **missing**. Run it on every new file, before anything else.
9. `groupby` = **split, apply, combine**. One line replaces a whole program.
10. `print(df.columns)` is your debugger. Most errors are a column name typed slightly wrong.

---

*Python Course · Day 7 · Pandas*